<a href="https://colab.research.google.com/github/Yousef-Shihade/Big-Data-HomeWork---Spark/blob/main/BigDataHW2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **<font color="brown">HomeWork 2 - Big Data - Winter25/26</font>**
---

### **Submitted by:**
### Yousef Shihade

### Shada Esawi








### Uni of Haifa

	****Upload the IMBD.csv dataset to the active Colab session storage before running the cells****

## **<font color="blue">Initialize Spark</font>**

In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

# Initialize Spark
spark = SparkSession.builder.appName("IMDB Big Data Analytics") \
    .master("local[*]").getOrCreate()

# Load the dataset
df = spark.read.csv("IMBD.csv", header=True, inferSchema=True, quote='"', escape='"')

# Peek at the initial data types before we start cleaning
print("Raw Data Schema:")
df.printSchema()

Raw Data Schema:
root
 |-- title: string (nullable = true)
 |-- year: string (nullable = true)
 |-- certificate: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- description: string (nullable = true)
 |-- stars: string (nullable = true)
 |-- votes: string (nullable = true)



## **<font color="blue">Run all imports</font>**

In [ ]:
from pyspark.sql.functions import (
    col, regexp_replace, regexp_extract, split, explode, trim, row_number,
    lower, count, avg, sum, desc, round, stddev, when
)
from pyspark.sql.window import Window
from pyspark.ml.feature import StopWordsRemover

## **<font color="blue">Task 1</font>**

In [ ]:
# 1. Drop rows with missing data in our most important columns
df_clean = df.dropna(subset=["title", "rating", "year", "genre", "votes"])

# 2. Clean 'votes' by removing commas and converting to an integer
df_clean = df_clean.withColumn("votes", regexp_replace(col("votes"), ",", "").cast("int"))

# 3. Clean 'duration' by stripping " min"
df_clean = df_clean.withColumn("duration", regexp_replace(col("duration"), " min", "").cast("int"))

# 4. Remove parentheses from the year string for easier parsing
df_clean = df_clean.withColumn("year_str", regexp_replace(col("year"), r"\(|\)", ""))

# 5. Extract the 4-digit starting year and drop any rows that fail to parse
df_clean = df_clean.withColumn("start_year", regexp_extract(col("year_str"), r"(\d{4})", 1).cast("int")) \
                   .filter(col("start_year").isNotNull())

# 6. Flag TV shows if the year contains a dash (e.g. "2018– ")
df_clean = df_clean.withColumn(
    "is_tv_show", col("year_str").rlike(r"\d{4}\s*[-–—]\s*\d{0,4}")
)

# Cache the clean dataset in memory to speed up the next tasks
df_clean.cache()

# Peek at the results to verify our cleaning logic
df_clean.select("title", "year", "year_str", "start_year", "rating", "votes", "is_tv_show").show(10, truncate=False)

+----------------------+-----------+---------+----------+------+-------+----------+
|title                 |year       |year_str |start_year|rating|votes  |is_tv_show|
+----------------------+-----------+---------+----------+------+-------+----------+
|Cobra Kai             |(2018– )   |2018–    |2018      |8.5   |177031 |true      |
|The Crown             |(2016– )   |2016–    |2016      |8.7   |199885 |true      |
|Better Call Saul      |(2015–2022)|2015–2022|2015      |8.9   |501384 |true      |
|Devil in Ohio         |(2022)     |2022     |2022      |5.9   |9773   |false     |
|Cyberpunk: Edgerunners|(2022– )   |2022–    |2022      |8.6   |15413  |true      |
|The Sandman           |(2022– )   |2022–    |2022      |7.8   |116358 |true      |
|Rick and Morty        |(2013– )   |2013–    |2013      |9.2   |502160 |true      |
|Breaking Bad          |(2008–2013)|2008–2013|2008      |9.5   |1831340|true      |
|The Imperfects        |(2022– )   |2022–    |2022      |6.3   |3123   |true

## **<font color="blue">Task 2</font>**

In [ ]:
# 1. Filter out TV shows so we only rank movies
movies_df = df_clean.filter(col("is_tv_show") == False)

# 2. Remove duplicate rows for the same movie (same title + same start_year)
movies_df = movies_df.dropDuplicates(["title", "start_year"])

# 3. Split comma-separated genres and create a new row for each genre
genre_df = movies_df.withColumn("single_genre", explode(split(col("genre"), r",\s*")))

# Clean extra spaces
genre_df = genre_df.withColumn("single_genre", trim(col("single_genre")))

# 4. Create a window to rank movies within each genre
# votes and title are used as "tie breakers" for consistent ordering
w = Window.partitionBy("single_genre").orderBy(
    col("rating").desc(), col("votes").desc(), col("title").asc()
)

# 5. Assign ranks and keep only the top 10 per genre
top_10_movies_by_genre = genre_df.withColumn("rank", row_number().over(w)).filter(col("rank") <= 10)

# 6. Count total rows so all genres appear
total_rows = top_10_movies_by_genre.count()

# Show all results
top_10_movies_by_genre.select(
    "single_genre", "rank", "title", "start_year", "rating", "votes"
).orderBy("single_genre", "rank").show(total_rows, truncate=False)

+------------+----+----------------------------------------------------------+----------+------+-------+
|single_genre|rank|title                                                     |start_year|rating|votes  |
+------------+----+----------------------------------------------------------+----------+------+-------+
|Action      |1   |The Lord of the Rings: The Return of the King             |2003      |9.0   |1819157|
|Action      |2   |The Lord of the Rings: The Fellowship of the Ring         |2001      |8.8   |1844055|
|Action      |3   |The Lord of the Rings: The Two Towers                     |2002      |8.8   |1642708|
|Action      |4   |Chen qing ling                                            |2019      |8.8   |6380   |
|Action      |5   |Mr. Sunshine                                              |2018      |8.7   |7620   |
|Action      |6   |Rurouni Kenshin: Trust and Betrayal                       |1999      |8.6   |14840  |
|Action      |7   |Lastman                             

## **<font color="blue">Task 3</font>**

In [ ]:
# 1. Start with rows that actually have actor data
base = df_clean.filter(col("stars").isNotNull())

# Clean up the raw text by removing brackets, quotes, and labels like "Director:"
stars_txt = regexp_replace(col("stars"), r"[\[\]\'\"]", "")
stars_txt = regexp_replace(stars_txt, r"(?i)stars?:", "")
stars_txt = regexp_replace(stars_txt, r"(?i)director:", "")
stars_txt = regexp_replace(stars_txt, r"\|", ",") # Convert pipes to commas so we can split easily

# Explode the actors into individual rows and trim any leftover whitespace
actors = base.select(
    "title",
    "start_year",  # Using year alongside title helps us separate movie remakes
    explode(split(stars_txt, r",\s*")).alias("actor")
).withColumn("actor", trim(col("actor")))

# Remove empty strings
actors = actors.filter(col("actor") != "")

# Ensure an actor is only listed once per movie to avoid double counting
actors = actors.select("title", "start_year", "actor").distinct()

# 2. Find actor collaborations by joining the movie data to itself
# We use < to ensure we don't match an actor with themselves or get duplicate pairs (A-B vs B-A)
pairs = actors.alias("a1").join(
    actors.alias("a2"),
    (col("a1.title") == col("a2.title")) & (col("a1.start_year") == col("a2.start_year"))
).filter(col("a1.actor") < col("a2.actor"))

# 3. Count the collaborations and sort to find the most frequent co-stars
collaborations = pairs.groupBy(
    col("a1.actor").alias("actor1"),
    col("a2.actor").alias("actor2")
).count().withColumnRenamed("count", "collaboration_count") \
 .orderBy(col("collaboration_count").desc())

# Show the results
collaborations.show(20, truncate=False)

+------------------+---------------+-------------------+
|actor1            |actor2         |collaboration_count|
+------------------+---------------+-------------------+
|Jan Suter         |Raúl Campos    |15                 |
|John Paul Tremblay|Robb Wells     |11                 |
|John Cleese       |Terry Gilliam  |10                 |
|John Paul Tremblay|Mike Smith     |9                  |
|Jakob Eklund      |Jens Hultén    |9                  |
|Mike Smith        |Robb Wells     |9                  |
|Eric Idle         |John Cleese    |9                  |
|Jakob Eklund      |Mikael Tornving|8                  |
|Dave Chappelle    |Stan Lathan    |8                  |
|Jakob Eklund      |Richard Holm   |7                  |
|Bruno Corbucci    |Tomas Milian   |7                  |
|Jakob Eklund      |Meliz Karlge   |7                  |
|Jakob Eklund      |Joel Kinnaman  |7                  |
|John Cleese       |Terry Jones    |7                  |
|Eric Idle         |Terry Gilli

## **<font color="blue">Task 4</font>**

In [ ]:
# Define our criterias
rating_threshold = 8.0
votes_threshold = 10000

# 1. Filter for movies that meet our criteria
hidden_gems_sorted = df_clean.filter(
    (~col("is_tv_show")) &                # Must be a movie
    (col("rating") > rating_threshold) &  # Must be highly rated
    (col("votes") < votes_threshold)      # Must have very few votes
).select(
    "title", "start_year", "genre", "rating", "votes"
).distinct().orderBy(
    # 2. Sort by the best rating first. If there's a tie,
    # the movie with the fewest votes (the most "hidden") wins
    col("rating").desc(),
    col("votes").asc()
)

# Display the top 20 hidden gems
hidden_gems_sorted.show(20, truncate=False)

+-------------------------------------+----------+-------------------------+------+-----+
|title                                |start_year|genre                    |rating|votes|
+-------------------------------------+----------+-------------------------+------+-----+
|Elesin Oba: The King's Horseman      |2022      |Adventure, Drama, History|9.4   |72   |
|Vagabond                             |2019      |Action, Mystery, Thriller|9.3   |187  |
|Julie and the Phantoms               |2020      |Adventure, Comedy, Drama |9.2   |595  |
|Julie and the Phantoms               |2020      |Adventure, Comedy, Drama |9.1   |546  |
|Hans Zimmer Live in Prague           |2017      |Documentary, Music       |9.1   |2710 |
|My Mister                            |2018      |Drama, Family            |9.1   |6442 |
|Surau dan Silek                      |2017      |Comedy, Drama, Family    |9.0   |71   |
|#HappyBirthdaySense8                 |2017      |Short                    |9.0   |844  |
|The Umbil

## **<font color="blue">Task 5</font>**

In [ ]:
# 1. Filter for movies only and grab unique titles
movies_only = df_clean.filter(col("is_tv_show") == False)
titles = movies_only.select("title").distinct()

# 2. Convert to lowercase and strip out all punctuation
clean_titles = titles.withColumn("clean_title", lower(regexp_replace(col("title"), r"[^\w\s]", "")))

# 3. Split the titles into an array of words
words_array_df = clean_titles.withColumn("words", split(col("clean_title"), r"\s+"))

# 4. Use PySpark's built-in StopWordsRemover
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
clean_words_array_df = remover.transform(words_array_df)

# 5. Now we explode the clean arrays into individual rows
exploded_words = clean_words_array_df.select(explode(col("filtered_words")).alias("word"))

# 6. Filter out empty strings and pure numbers
final_words = exploded_words.filter((col("word") != "") & (~col("word").rlike(r"^\d+$")))

# 7. Count and sort
word_counts = final_words.groupBy("word").count().withColumnRenamed("count", "frequency") \
    .orderBy(col("frequency").desc())

word_counts.show(20, truncate=False)

+---------+---------+
|word     |frequency|
+---------+---------+
|love     |87       |
|christmas|52       |
|man      |45       |
|live     |42       |
|de       |41       |
|story    |41       |
|one      |41       |
|movie    |38       |
|life     |37       |
|girl     |33       |
|world    |32       |
|la       |31       |
|last     |31       |
|night    |30       |
|house    |30       |
|king     |30       |
|happy    |29       |
|little   |28       |
|black    |26       |
|american |26       |
+---------+---------+
only showing top 20 rows


## **<font color="blue">Task 6</font>**

In [ ]:
# 1. Filter for movies only
movies_df = df_clean.filter(col("is_tv_show") == False)

# Split and explode the genres
df_genres = movies_df.withColumn("single_genre", explode(split(col("genre"), r",\s*")))

# Clean up spaces
df_genres = df_genres.withColumn("single_genre", trim(col("single_genre")))

# 2. Filter out empty or null genres
df_genres_valid = df_genres.filter(col("single_genre").isNotNull() &
    (col("single_genre") != "null") & (col("single_genre") != ""))

# 3. Group by genre and calculate the variability of ratings
genre_diversity = df_genres_valid.groupBy("single_genre") \
    .agg(stddev("rating").alias("variability_stddev")) \
    .filter(col("variability_stddev").isNotNull())

# Sort from highest variability to lowest
genre_diversity_sorted = genre_diversity.orderBy(col("variability_stddev").desc())
genre_diversity_sorted.show(100, truncate=False)

# 4. Extract the highest and lowest variability
highest_var = genre_diversity_sorted.first()
lowest_var = genre_diversity.orderBy(col("variability_stddev").asc()).first()

# Print a summary
print(f"Highest variability: '{highest_var['single_genre']}' with {highest_var['variability_stddev']:.4f}")
print(f"Lowest variability:  '{lowest_var['single_genre']}' with {lowest_var['variability_stddev']:.4f}")

+------------+-------------------+
|single_genre|variability_stddev |
+------------+-------------------+
|Horror      |1.3742688503065574 |
|Game-Show   |1.3499559075691892 |
|Musical     |1.3414194131540995 |
|Thriller    |1.3073122627605618 |
|Action      |1.286768723255537  |
|Sci-Fi      |1.2733534108146312 |
|Fantasy     |1.2382859868877605 |
|Family      |1.2118787360276049 |
|Mystery     |1.204085512853544  |
|Reality-TV  |1.196092121718208  |
|Adventure   |1.1607526155318058 |
|Comedy      |1.156600313533113  |
|Romance     |1.1137710066758002 |
|Drama       |1.0875001719743451 |
|War         |1.0854471376911015 |
|Crime       |1.0841886924940765 |
|Short       |1.0708463224893405 |
|Talk-Show   |1.0456789394668158 |
|Animation   |1.0363284546935452 |
|Music       |0.9919521262543951 |
|News        |0.9311599344524074 |
|Sport       |0.9155594094639982 |
|Biography   |0.8734825421337453 |
|Documentary |0.8515051882772872 |
|History     |0.8102202318713052 |
|Western     |0.5663

## **<font color="blue">Task 7</font>**

In [ ]:
# 1. Filter out TV shows so we only analyze movies
movies_only = df_clean.filter(col("is_tv_show") == False)

# 2. Clean up the data by dropping rows with missing certificates
movies_valid = movies_only.filter(col("certificate").isNotNull() & (col("certificate") != "")
)

# 3. Group by certification to count the movies and calculate their average rating
certification_dist = movies_valid.groupBy("certificate").agg(
    count("*").alias("movie_count"), avg("rating").alias("avg_rating"))

# Show the distribution, ordered by the most common certificates first
certification_dist.orderBy(desc("movie_count")).show(50, truncate=False)

# 4. Find the certification with the highest average rating
highest_cert = certification_dist.orderBy(desc("avg_rating")).first()

# Print the results
print(f"\nCertification with highest average rating: "
      f"{highest_cert['certificate']} "
      f"(avg_rating = {highest_cert['avg_rating']:.3f})")

+-----------+-----------+------------------+
|certificate|movie_count|avg_rating        |
+-----------+-----------+------------------+
|TV-MA      |1300       |6.52230769230768  |
|TV-14      |502        |6.4338645418326745|
|R          |466        |6.038197424892706 |
|Not Rated  |442        |6.102714932126699 |
|PG-13      |276        |6.362318840579713 |
|TV-PG      |169        |6.508284023668639 |
|PG         |153        |6.44313725490196  |
|TV-G       |97         |6.95257731958763  |
|Unrated    |62         |6.1822580645161285|
|TV-Y7      |59         |6.7372881355932215|
|G          |43         |6.790697674418604 |
|TV-Y       |31         |6.17741935483871  |
|Approved   |23         |6.7826086956521765|
|TV-Y7-FV   |7          |6.685714285714285 |
|Passed     |6          |6.866666666666666 |
|NC-17      |4          |6.2               |
|M          |1          |5.8               |
|MA-17      |1          |4.7               |
|12         |1          |7.4               |
+---------

## **<font color="blue">Task 8</font>**

In [ ]:
# Create category column
df_with_category = df_clean.withColumn(
    "category", when(col("is_tv_show"), "TV Show").otherwise("Movie"))

# 1 & 2. Compare TV Shows vs Movies
tv_vs_movies = df_with_category.groupBy("category").agg(
    count("*").alias("total_titles"),
    round(avg("rating"), 2).alias("avg_rating"),
    round(avg("votes"), 0).alias("avg_votes"),
    sum("votes").alias("total_votes")
)

print("\n--- TV Shows vs Movies ---")
tv_vs_movies.select("category","total_titles","avg_rating","avg_votes","total_votes") \
    .orderBy("category").show(truncate=False)

# 3. Trends over time
trends = df_with_category.groupBy("start_year", "category").agg(
    count("*").alias("num_titles"),
    round(avg("rating"), 2).alias("avg_rating"),
    sum("votes").alias("total_popularity_votes")
).orderBy(desc("start_year"))

print("\n--- Trends over time (Newest to Oldest) ---")
trends.show(50, truncate=False)


--- TV Shows vs Movies ---
+--------+------------+----------+---------+-----------+
|category|total_titles|avg_rating|avg_votes|total_votes|
+--------+------------+----------+---------+-----------+
|Movie   |5419        |6.4       |20547.0  |111343150  |
|TV Show |3353        |7.35      |17938.0  |60147503   |
+--------+------------+----------+---------+-----------+


--- Trends over time (Newest to Oldest) ---
+----------+--------+----------+----------+----------------------+
|start_year|category|num_titles|avg_rating|total_popularity_votes|
+----------+--------+----------+----------+----------------------+
|2022      |TV Show |207       |6.74      |869408                |
|2022      |Movie   |367       |6.12      |2744673               |
|2021      |TV Show |282       |6.83      |2217121               |
|2021      |Movie   |557       |6.35      |5964817               |
|2020      |Movie   |752       |6.54      |6564756               |
|2020      |TV Show |362       |7.03      |16032

## **<font color="blue">Task 9</font>**

In [ ]:
# 1. Filter out TV shows and remove any rows with missing/empty certificates
movies_valid = df_clean.filter(
    (col("is_tv_show") == False) & col("certificate").isNotNull() & (col("certificate") != ""))

# 2 & 3. Group by certification, count the movies, and calculate the average rating
cert_stats = movies_valid.groupBy("certificate").agg(
    count("*").alias("total_movies"), round(avg("rating"), 2).alias("avg_rating"))

cert_stats.orderBy(desc("total_movies")).show(50, truncate=False)

# 4. Find the certification with the highest average rating
best_cert = cert_stats.orderBy(desc("avg_rating")).first()

# Print the results
print(f"\nCertification with the highest average rating: "
      f"{best_cert['certificate']} (avg_rating = {best_cert['avg_rating']:.2f})")

+-----------+------------+----------+
|certificate|total_movies|avg_rating|
+-----------+------------+----------+
|TV-MA      |1300        |6.52      |
|TV-14      |502         |6.43      |
|R          |466         |6.04      |
|Not Rated  |442         |6.1       |
|PG-13      |276         |6.36      |
|TV-PG      |169         |6.51      |
|PG         |153         |6.44      |
|TV-G       |97          |6.95      |
|Unrated    |62          |6.18      |
|TV-Y7      |59          |6.74      |
|G          |43          |6.79      |
|TV-Y       |31          |6.18      |
|Approved   |23          |6.78      |
|TV-Y7-FV   |7           |6.69      |
|Passed     |6           |6.87      |
|NC-17      |4           |6.2       |
|M          |1           |5.8       |
|MA-17      |1           |4.7       |
|12         |1           |7.4       |
+-----------+------------+----------+


Certification with the highest average rating: 12 (avg_rating = 7.40)


## **<font color="blue">Task 10</font>**

In [ ]:
# 1. Overall comparison (TV vs Movies)
overall = df_clean.groupBy("is_tv_show").agg(
    count("*").alias("total_titles"),
    round(avg("rating"), 2).alias("avg_rating"),
    sum("votes").alias("total_votes")
).withColumn(
    "category", when(col("is_tv_show"), "TV Show").otherwise("Movie"))

print("--- Overall comparison (TV vs Movies) ---")
overall.select(
    "category", "total_titles", "avg_rating", "total_votes"
).orderBy(desc("total_votes")).show(truncate=False)

# 2. Trends over time (ratings + popularity votes per year)
trends_over_time = df_clean.groupBy("start_year", "is_tv_show").agg(
    round(avg("rating"), 2).alias("avg_rating"),
    sum("votes").alias("total_votes_per_year"),
    count("*").alias("release_count")
).withColumn("category", when(col("is_tv_show"), "TV Show").otherwise("Movie"))

print("--- Trends over time (Newest to Oldest) ---")
trends_over_time.select(
    "start_year", "category", "avg_rating", "total_votes_per_year", "release_count"
).orderBy(desc("start_year")).filter(col("start_year") >= 2000).show(50, truncate=False)

--- Overall comparison (TV vs Movies) ---
+--------+------------+----------+-----------+
|category|total_titles|avg_rating|total_votes|
+--------+------------+----------+-----------+
|Movie   |5419        |6.4       |111343150  |
|TV Show |3353        |7.35      |60147503   |
+--------+------------+----------+-----------+

--- Trends over time (Newest to Oldest) ---
+----------+--------+----------+--------------------+-------------+
|start_year|category|avg_rating|total_votes_per_year|release_count|
+----------+--------+----------+--------------------+-------------+
|2022      |Movie   |6.12      |2744673             |367          |
|2022      |TV Show |6.74      |869408              |207          |
|2021      |Movie   |6.35      |5964817             |557          |
|2021      |TV Show |6.83      |2217121             |282          |
|2020      |TV Show |7.03      |1603276             |362          |
|2020      |Movie   |6.54      |6564756             |752          |
|2019      |TV Show